# Session 2 — Local NLI model smoke test

Proves that `cross-encoder/nli-deberta-v3-small` runs fully locally (weights cached in
`~/.cache/huggingface`) — no paid APIs involved.

Run from `backend/` with the `TruthLens (venv backend)` kernel selected — the global Python 3.12
on this machine has no `torch`. Register the kernel once with:

```bash
.venv\Scripts\python.exe -m ipykernel install --user --name truthlens-venv --display-name "TruthLens (venv backend)"
```

In [1]:
from transformers import pipeline

# This downloads the open-source NLI model locally (cached in ~/.cache/huggingface)
nli_model = pipeline("text-classification", model="cross-encoder/nli-deberta-v3-small")

# Test claim and evidence
claim = "U-Net was introduced in 2015."
evidence = "U-Net: Convolutional Networks for Biomedical Image Segmentation was published by Ronneberger et al. at MICCAI 2015."

# Run local inference
result = nli_model({"text": claim, "text_pair": evidence})
print(result)

{'label': 'neutral', 'score': 0.9968330264091492}


## Why that prints `neutral`, not `entailment`

The pipeline's dict form is `{"text": <premise>, "text_pair": <hypothesis>}` — and in NLI the
**evidence is the premise**, while the **claim is the hypothesis** being checked. The cell above
has them swapped.

The model is not broken: the label order is `{0: contradiction, 1: entailment, 2: neutral}` and a
flipped pair changes the verdict (see below). This is a real constraint for `verifier.py`: always
call it as `("text": evidence, "text_pair": claim)`.

In [2]:
# Correct order: evidence is the premise, the claim is the hypothesis.
result = nli_model({"text": evidence, "text_pair": claim})
print(result)

# Sanity check: the model must separate entailment from contradiction.
sanity_pairs = [
    ("A cat is sitting on the mat.", "The cat sat on the mat."),
    ("The cat is on the mat.", "There is no cat on the mat."),
    ("The Eiffel Tower is in Paris.", "The Eiffel Tower is located in Berlin."),
]
for premise, hypothesis in sanity_pairs:
    out = nli_model({"text": premise, "text_pair": hypothesis})
    print(f"{out['label']:14} {out['score']:.3f}  |  {premise} -> {hypothesis}")

{'label': 'entailment', 'score': 0.8248386383056641}


entailment     0.993  |  A cat is sitting on the mat. -> The cat sat on the mat.


contradiction  1.000  |  The cat is on the mat. -> There is no cat on the mat.


contradiction  1.000  |  The Eiffel Tower is in Paris. -> The Eiffel Tower is located in Berlin.


## Result

Local, offline NLI classification works end to end. Two things to carry into the pipeline:

1. Call the model as `(evidence, claim)`, never `(claim, evidence)`.
2. Map logits with `{0: contradiction, 1: entailment, 2: neutral}` — this is what `verifier.py`
   will consume in Session 3.

Next: download FEVER and build the evidence corpus.